# ContractRisk — juez con respaldo literal y control de repetición

Protocolo nuevo posterior a inspeccionar los fallos de la corrida 04. Mismo Qwen3-8B NF4, revisión, presupuesto y muestreo. No es validación ciega ni permite atribuir mejoras a un cambio aislado.

El juez devuelve citas textuales por campo para respuesta y explicación. El programa comprueba que existan en fragmentos citados, pero no garantiza su pertinencia semántica. Una nota incorrecta se conserva sin corregirla. Solo para la condición mitigada se colapsan bloques completos repetidos exactamente; no hay resumen ni eliminación de contenido distinto. Las fuentes y citas no se normalizan.

Primero evalúa nueve casos piloto (los siete anteriores más atribución falsa y conclusión prudente). Se detiene al primer fallo. Si pasan los nueve, continúa hasta 38 juicios reutilizando esas notas. Mantiene los umbrales numéricos y exige respaldo literal válido. La igualdad de entradas normalizadas en pares repetidos no demuestra ausencia general de sesgo.

Sesión nueva T4; habilitar acceso a WANDB_API_KEY y ejecutar en orden. Sin cambios en main ni Space. No conserva razonamiento privado. Confirmación nueva y auditoría humana siguen pendientes aunque supere regresión.


In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_literal_judge_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-calibration.txt'], 'installation.log')
import torch
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('20 anclas aprobadas; hashes y rúbrica comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
RESULTS = ROOT / 'literal_judge_results'
run_visible([sys.executable, '-u', 'final/run_literal_judge.py', '--output', str(RESULTS), '--require-wandb'], 'literal_judge.log')


## Inspección y conservación

completed_pilot_failed puede contener menos de nueve juicios: es una detención temprana deliberada, no una corrida completa. completed_regression_failed significa que completó más casos pero no superó el filtro. completed_regression_passed_confirmation_pending todavía no autoriza al juez final.

No repetir para elegir notas favorables. Revisar errores de generación, presupuesto, verificaciones y razones por separado. Conservar evidencia también si falló la carga.


In [ ]:
RESULTS = ROOT / 'literal_judge_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
shutil.copy2(ROOT / 'final/judge_literal_config.json', RESULTS / 'judge_literal_config.json')
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez_literal_resultados'), 'zip', RESULTS)
files.download(package)
